# OP-06 · NMME : téléchargement et valeurs par période

**Notebook opérationnel** — à exécuter chaque mois.

| | |
|---|---|
| Étape du workflow | E2 → entrée de E4 et E5 |
| Source | serveur NOAA/CPC, `International/nmme/netcdf/<mois><année>ic/` |
| Sorties | `DATA_OSF/raw/nmme/YYYYMM/` puis `DATA_OSF/derived/nmme/YYYYMM/` |
| Durée | environ 10 min (≈ 730 fichiers de 2,5 Mo, 6 en parallèle) |

**À savoir :** ces fichiers ne contiennent que la **moyenne d'ensemble**, au pas mensuel. NMME alimente donc les mois et les saisons, et seulement les méthodes de calibration fondées sur la moyenne d'ensemble. Sa SST (`tmpsfc`) servira de prédicteur en phase P4.

Équivalent en ligne de commande :
```
python scripts/run_download_nmme.py --config config/cycle_YYYYMM.yaml
python scripts/run_nmme_totals.py   --config config/cycle_YYYYMM.yaml
```

## Paramètres

In [ ]:
CYCLE_CONFIG = "config/cycle_202609.yaml"
MODELS       = None    # None = tous les modèles de la configuration
VARIABLES    = None    # None = prate, tmp2m, tmpsfc
WORKERS      = 6       # fichiers téléchargés simultanément

In [ ]:
from pathlib import Path
import os
import pandas as pd

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(REPO)
from eccas_s2s.settings import load_cycle
cfg = load_cycle(CYCLE_CONFIG)
print(f"Cycle {cfg.cycle_id} — initialisation {cfg.init_date.date()}")

## 1. Téléchargement

In [ ]:
from eccas_s2s.operations import download_nmme
dl = download_nmme.run(CYCLE_CONFIG, models=MODELS, variables=VARIABLES, workers=WORKERS)
print(f"\nStatut : {dl.status} — {len(dl.parameters.get('failures', {}))} échec(s)")
for w in dl.warnings[:10]:
    print(" ⚠", w)
display(pd.read_csv(cfg.raw_dir("nmme") / "nmme_download_summary.csv"))

## 2. Valeurs par mois et par saison

In [ ]:
from eccas_s2s.operations import nmme_totals
ctx = nmme_totals.run(CYCLE_CONFIG, models=MODELS)
print(f"\nStatut : {ctx.status}")
for w in ctx.warnings:
    print(" ⚠", w)
display(pd.read_csv(nmme_totals.derived_dir(cfg) / "nmme_totals_summary.csv"))

## 3. Contrôle visuel : cumul de la première saison prévue

In [ ]:
from eccas_s2s.viz.maps import map_panel
fields, titles = [], []
for m in cfg.raw["systems"]["nmme"]["models"]:
    try:
        da = nmme_totals.load_totals(cfg, m, "precip", "forecast").sel(period="season_m0").isel(year=0)
    except (FileNotFoundError, KeyError):
        continue
    fields.append(da); titles.append(m)
if fields:
    fig = map_panel(fields, titles, shapefile=cfg.raw["paths"]["shapefile"], cmap="YlGnBu",
                    levels=[0, 50, 100, 200, 300, 400, 500, 700, 900, 1200], extend="max", ncols=3,
                    cbar_label="mm", suptitle="NMME — cumul de la première saison (moyenne d'ensemble)")

In [ ]:
for c in (dl, ctx):
    print(f"{c.step:20s} {c.status:8s} {c.run_dir / 'manifest.json'}")